# AN2DL Dataset Cleaner - ENHANCED
## Detect and Remove Shrek & Slime Images

**Enhanced detection methods from Project.ipynb:**
- Better tissue mask detection (HSV-based background removal)
- LAB color space for Shrek detection (more accurate skin tone detection)
- Separate cold/warm outlier fractions
- Pink tissue guard for edge cases
- Fluorescent green detection for slime

**Output:**
- `Dataset_Clean/` - Cleaned dataset folder
- `Dataset_Clean/train_data/` - Cleaned training images and masks
- `Dataset_Clean/test_data/` - Copy of test data (unchanged)
- `Dataset_Clean/train_labels.csv` - Filtered labels
- `Dataset_Clean/removed_images.csv` - List of removed images

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount("/gdrive")

In [ ]:
import os
import numpy as np
import pandas as pd
import cv2
import shutil
from tqdm import tqdm
import matplotlib.pyplot as plt
from concurrent.futures import ThreadPoolExecutor, as_completed
import warnings
from datetime import datetime
import json
warnings.filterwarnings('ignore')

# Check GPU
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# =============================================================================
# CONFIGURATION
# =============================================================================

BASE_DIR = "/gdrive/MyDrive/AN2DL_Chall2/"
ORIGINAL_DATASET_DIR = os.path.join(BASE_DIR, "Dataset")
CLEAN_DATASET_DIR = os.path.join(BASE_DIR, "Dataset_Clean_v2")

# Original paths
ORIGINAL_TRAIN_DIR = os.path.join(ORIGINAL_DATASET_DIR, "train_data")
ORIGINAL_TEST_DIR = os.path.join(ORIGINAL_DATASET_DIR, "test_data")
ORIGINAL_LABELS_CSV = os.path.join(ORIGINAL_DATASET_DIR, "train_labels.csv")

# Clean paths
CLEAN_TRAIN_DIR = os.path.join(CLEAN_DATASET_DIR, "train_data")
CLEAN_TEST_DIR = os.path.join(CLEAN_DATASET_DIR, "test_data")
CLEAN_LABELS_CSV = os.path.join(CLEAN_DATASET_DIR, "train_labels.csv")
REMOVED_CSV = os.path.join(CLEAN_DATASET_DIR, "removed_images.csv")

# Processing settings
NUM_WORKERS = 8

# Manual false positive corrections (IDs that should NOT be removed)
FALSE_POSITIVE_IDS = {'0812', '1065'}

print(f"Original dataset: {ORIGINAL_DATASET_DIR}")
print(f"Clean dataset: {CLEAN_DATASET_DIR}")

In [ ]:
# Verify original dataset exists
assert os.path.exists(ORIGINAL_TRAIN_DIR), f"Train dir not found: {ORIGINAL_TRAIN_DIR}"
assert os.path.exists(ORIGINAL_TEST_DIR), f"Test dir not found: {ORIGINAL_TEST_DIR}"
assert os.path.exists(ORIGINAL_LABELS_CSV), f"Labels CSV not found: {ORIGINAL_LABELS_CSV}"

# Load original labels
original_labels_df = pd.read_csv(ORIGINAL_LABELS_CSV)
print(f"Original training samples: {len(original_labels_df)}")
print(f"\nClass distribution:")
print(original_labels_df['label'].value_counts())

## 1.Enhanced Detection Functions (from Project.ipynb)

In [ ]:
# ENHANCED TISSUE MASK DETECTION

def get_tissue_mask_hsv(img_rgb):
    """
    Enhanced tissue mask detection using HSV color space.
    Properly handles white/light gray backgrounds.

    Args:
        img_rgb: RGB image as numpy array (uint8 0-255 or float 0-1)

    Returns:
        tissue_mask: Boolean mask where True = tissue
        hsv_channels: Tuple of (H, S, V) channels
        img_uint8: Image converted to uint8
    """
    # Handle both float [0,1] and uint8 [0,255] inputs
    if img_rgb.dtype == np.float32 or img_rgb.dtype == np.float64:
        if img_rgb.max() <= 1.0:
            img_uint8 = (np.clip(img_rgb, 0, 1) * 255).astype(np.uint8)
        else:
            img_uint8 = np.clip(img_rgb, 0, 255).astype(np.uint8)
    else:
        img_uint8 = img_rgb.astype(np.uint8)

    # Convert to HSV
    hsv = cv2.cvtColor(img_uint8, cv2.COLOR_RGB2HSV)
    H, S, V = hsv[..., 0], hsv[..., 1], hsv[..., 2]

    # Background detection (white and light gray)
    white_bg = (V >= 230) & (S <= 25)
    light_gray_bg = (V >= 180) & (S <= 22)
    background = white_bg | light_gray_bg

    # Tissue = not background, with minimum saturation and value
    tissue_mask = (~background) & (S > 10) & (V > 26)

    return tissue_mask, (H, S, V), img_uint8

In [ ]:
# ENHANCED SLIME DETECTION

def classify_slime(img_rgb,
                   fluo_h_min=40, fluo_h_max=85,
                   fluo_s_min=60, fluo_v_min=180,
                   min_fluo_frac=0.01):
    """
    Enhanced slime detection using fluorescent green color detection.

    Slime images have bright fluorescent green patches that are:
    - High saturation (S >= 60)
    - High value/brightness (V >= 180)
    - Green hue range (H: 40-85)

    Args:
        img_rgb: RGB image
        fluo_h_min/max: Hue range for fluorescent green
        fluo_s_min: Minimum saturation
        fluo_v_min: Minimum value (brightness)
        min_fluo_frac: Minimum fraction of tissue that must be fluorescent

    Returns:
        dict with detection results
    """
    tissue_mask, (H, S, V), _ = get_tissue_mask_hsv(img_rgb)

    if tissue_mask.sum() == 0:
        return {
            "fluo_frac": 0.0,
            "is_slime": False,
            "reason": "No tissue detected"
        }

    # Detect fluorescent green
    green_fluo = (
        (H >= fluo_h_min) & (H <= fluo_h_max) &
        (S >= fluo_s_min) & (V >= fluo_v_min)
    )

    # Calculate fraction of tissue that is fluorescent green
    fluo_frac = (green_fluo & tissue_mask).sum() / tissue_mask.sum()

    is_slime = fluo_frac >= min_fluo_frac

    return {
        "fluo_frac": float(fluo_frac),
        "is_slime": bool(is_slime),
        "reason": f"Fluorescent green fraction: {fluo_frac:.3f}" if is_slime else ""
    }

In [ ]:
# ENHANCED SHREK DETECTION

def classify_shrek(img_rgb,
                   pink_guard=0.10,
                   min_pink_frac=0.04,
                   min_outlier_frac=0.10,
                   strong_cold_frac=0.20,
                   strong_warm_frac=0.14):
    """
    Enhanced Shrek detection using both LAB and HSV color spaces.

    Key insight: Real tissue images have pink/purple coloring from H&E staining.
    Shrek images have green/yellow/orange/brown "outlier" colors.

    Detection logic:
    1.If image has sufficient pink tissue (pink_guard), it's likely real
       - But still check for warm colors (orange/brown) which indicate Shrek
    2.If image lacks pink tissue:
       - High cold colors (green/yellow) = Shrek
       - High warm colors (orange/brown) = Shrek
       - High outlier + low pink = Shrek

    Args:
        img_rgb: RGB image
        pink_guard: If pink_frac >= this, apply relaxed detection
        min_pink_frac: Minimum pink to consider image as potentially real
        min_outlier_frac: Minimum outlier fraction to flag as Shrek
        strong_cold_frac: Strong green/yellow = definitely Shrek
        strong_warm_frac: Strong orange/brown = definitely Shrek

    Returns:
        dict with detection results
    """
    tissue_mask, (H, S, V), img_uint8 = get_tissue_mask_hsv(img_rgb)

    if tissue_mask.sum() == 0:
        return {
            "pink_frac": 0.0,
            "cold_frac": 0.0,
            "warm_frac": 0.0,
            "outlier_frac": 0.0,
            "is_shrek": True,
            "reason": "No tissue detected"
        }

    # Convert to LAB color space for better color discrimination
    lab = cv2.cvtColor(img_uint8, cv2.COLOR_RGB2LAB)
    A = lab[..., 1].astype(np.int16)  # Green-Red axis
    B = lab[..., 2].astype(np.int16)  # Blue-Yellow axis

    # =========================
    # PINK/PURPLE TISSUE DETECTION (H&E staining colors)
    # =========================

    # Strong pink (high A value = red/magenta)
    pink_strong = (A >= 142) & (S >= 12)

    # Lavender/light pink
    pink_lavender = (A >= 130) & (A <= 142) & (B >= 128) & (B <= 155) & (S >= 6)

    # Desaturated purple in LAB space
    purple_desat_lab = (A >= 128) & (A <= 145) & (B >= 108) & (B <= 130) & (S >= 5)

    # Soft purple in HSV space
    purple_soft_hsv = (H >= 125) & (H <= 165) & (S >= 8) & (V > 30)

    pink_mask = pink_strong | pink_lavender | purple_desat_lab | purple_soft_hsv
    pink_frac = (pink_mask & tissue_mask).sum() / tissue_mask.sum()

    # =========================
    # SHREK COLOR DETECTION (outlier colors)
    # =========================

    # Cold colors: Green and yellow (Shrek's skin)
    green_yellow = (H >= 30) & (H <= 88) & (S >= 58) & (V >= 55)
    green_fluo = (H >= 40) & (H <= 85) & (S >= 40) & (V >= 160)

    # Warm colors: Orange, gold, brown, mustard (Shrek's clothes, background)
    orange_gold = (H >= 14) & (H <= 34) & (S >= 32) & (V >= 40)
    brown_mustard = (H >= 6) & (H <= 34) & (S >= 24) & (V >= 30) & (V < 230)

    # =========================
    # DETECTION LOGIC
    # =========================

    if pink_frac >= pink_guard:
        # Image has substantial pink tissue - likely real
        # But still check for strong warm colors (Shrek mixed with tissue)
        cold_mask = green_yellow
        warm_mask = orange_gold | brown_mustard

        cold_frac = (cold_mask & tissue_mask).sum() / tissue_mask.sum()
        warm_frac = (warm_mask & tissue_mask).sum() / tissue_mask.sum()
        outlier_frac = ((cold_mask | warm_mask) & tissue_mask).sum() / tissue_mask.sum()

        # Only flag as Shrek if very strong warm colors
        is_shrek = warm_frac >= strong_warm_frac

        reason = ""
        if is_shrek:
            reason = f"High warm colors despite pink tissue: warm={warm_frac:.3f}"

        return {
            "pink_frac": float(pink_frac),
            "cold_frac": float(cold_frac),
            "warm_frac": float(warm_frac),
            "outlier_frac": float(outlier_frac),
            "is_shrek": bool(is_shrek),
            "reason": reason
        }

    # Image lacks pink tissue - apply stricter detection
    cold_mask = green_yellow | green_fluo
    warm_mask = orange_gold | brown_mustard

    cold_frac = (cold_mask & tissue_mask).sum() / tissue_mask.sum()
    warm_frac = (warm_mask & tissue_mask).sum() / tissue_mask.sum()
    outlier_frac = ((cold_mask | warm_mask) & tissue_mask).sum() / tissue_mask.sum()

    # Multiple conditions for Shrek detection
    is_shrek = (
        (cold_frac >= strong_cold_frac) or  # Very green/yellow
        (warm_frac >= strong_warm_frac) or  # Very orange/brown
        ((outlier_frac >= min_outlier_frac) and (pink_frac < min_pink_frac))  # Outlier + no pink
    )

    reason = ""
    if is_shrek:
        reasons = []
        if cold_frac >= strong_cold_frac:
            reasons.append(f"High cold colors: {cold_frac:.3f}")
        if warm_frac >= strong_warm_frac:
            reasons.append(f"High warm colors: {warm_frac:.3f}")
        if (outlier_frac >= min_outlier_frac) and (pink_frac < min_pink_frac):
            reasons.append(f"Outlier + low pink: outlier={outlier_frac:.3f}, pink={pink_frac:.3f}")
        reason = "; ".join(reasons)

    return {
        "pink_frac": float(pink_frac),
        "cold_frac": float(cold_frac),
        "warm_frac": float(warm_frac),
        "outlier_frac": float(outlier_frac),
        "is_shrek": bool(is_shrek),
        "reason": reason
    }

print("Enhanced detection functions defined!")

In [ ]:
# COMBINED ANOMALY DETECTION

def extract_id_from_filename(filename):
    """Extract numeric ID from filename like 'img_0123.png' -> '0123'"""
    import re
    match = re.search(r'img_(\d+)', filename)
    return match.group(1) if match else filename


def detect_anomaly(img_rgb, sample_id=None):
    """
    Combined anomaly detection for both Slime and Shrek.

    Args:
        img_rgb: RGB image (uint8 or float)
        sample_id: Optional sample ID for false positive checking

    Returns:
        dict with detection results
    """
    # Check for manual false positive override
    if sample_id and sample_id in FALSE_POSITIVE_IDS:
        return {
            "is_anomalous": False,
            "anomaly_type": None,
            "reason": "Manual false positive override",
            "slime_result": {},
            "shrek_result": {}
        }

    # First check for slime (bright fluorescent green)
    slime_result = classify_slime(img_rgb)
    if slime_result["is_slime"]:
        return {
            "is_anomalous": True,
            "anomaly_type": "slime",
            "reason": slime_result.get("reason", "Slime detected"),
            "slime_result": slime_result,
            "shrek_result": {}
        }

    # Then check for Shrek
    shrek_result = classify_shrek(img_rgb)
    if shrek_result["is_shrek"]:
        return {
            "is_anomalous": True,
            "anomaly_type": "shrek",
            "reason": shrek_result.get("reason", "Shrek detected"),
            "slime_result": slime_result,
            "shrek_result": shrek_result
        }

    # Not anomalous
    return {
        "is_anomalous": False,
        "anomaly_type": None,
        "reason": "",
        "slime_result": slime_result,
        "shrek_result": shrek_result
    }


def process_single_image(args):
    """Process a single image for anomaly detection (parallel processing)."""
    sample_index, data_dir = args
    sample_id = extract_id_from_filename(sample_index)

    try:
        image_path = os.path.join(data_dir, sample_index)
        image = cv2.imread(image_path)

        if image is None:
            return {
                "sample_index": sample_index,
                "sample_id": sample_id,
                "is_anomalous": True,
                "anomaly_type": "error",
                "reason": "Could not load image",
                "stats": {}
            }

        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        result = detect_anomaly(image_rgb, sample_id)

        # Compile stats
        stats = {}
        if result.get("slime_result"):
            stats["fluo_frac"] = result["slime_result"].get("fluo_frac", 0)
        if result.get("shrek_result"):
            stats.update({
                "pink_frac": result["shrek_result"].get("pink_frac", 0),
                "cold_frac": result["shrek_result"].get("cold_frac", 0),
                "warm_frac": result["shrek_result"].get("warm_frac", 0),
                "outlier_frac": result["shrek_result"].get("outlier_frac", 0),
            })

        return {
            "sample_index": sample_index,
            "sample_id": sample_id,
            "is_anomalous": result["is_anomalous"],
            "anomaly_type": result["anomaly_type"],
            "reason": result["reason"],
            "stats": stats
        }

    except Exception as e:
        return {
            "sample_index": sample_index,
            "sample_id": sample_id,
            "is_anomalous": True,
            "anomaly_type": "error",
            "reason": f"Error: {str(e)}",
            "stats": {}
        }

print("Combined detection functions defined!")

## 2. Scan Dataset for Anomalies

In [ ]:
# Prepare arguments for parallel processing
sample_indices = original_labels_df['sample_index'].tolist()
args_list = [(idx, ORIGINAL_TRAIN_DIR) for idx in sample_indices]

print(f"Scanning {len(sample_indices)} images for anomalies...")
print(f"Using {NUM_WORKERS} workers for parallel processing")
print(f"False positive overrides: {FALSE_POSITIVE_IDS}")

# Process images in parallel
results = []
with ThreadPoolExecutor(max_workers=NUM_WORKERS) as executor:
    futures = {executor.submit(process_single_image, args): args[0] for args in args_list}

    for future in tqdm(as_completed(futures), total=len(futures), desc="Scanning"):
        result = future.result()
        results.append(result)

print(f"\nScanning complete! ")

In [ ]:
# Analyze results
anomalous_results = [r for r in results if r['is_anomalous']]
valid_results = [r for r in results if not r['is_anomalous']]

shrek_results = [r for r in anomalous_results if r['anomaly_type'] == 'shrek']
slime_results = [r for r in anomalous_results if r['anomaly_type'] == 'slime']
error_results = [r for r in anomalous_results if r['anomaly_type'] == 'error']

print("="*60)
print("SCAN RESULTS (Enhanced Detection)")
print("="*60)
print(f"Total images scanned: {len(results)}")
print(f"Valid images: {len(valid_results)}")
print(f"Anomalous images: {len(anomalous_results)} ({100*len(anomalous_results)/len(results):.2f}%)")
print(f"  - Shrek: {len(shrek_results)}")
print(f"  - Slime: {len(slime_results)}")
print(f"  - Errors: {len(error_results)}")

In [ ]:
# Visualize detected anomalies
def visualize_anomalies(anomaly_list, title, max_show=8):
    """Visualize detected anomalous images."""
    if len(anomaly_list) == 0:
        print(f"No {title} images detected! ")
        return

    n_show = min(len(anomaly_list), max_show)
    cols = min(4, n_show)
    rows = (n_show + cols - 1) // cols

    fig, axes = plt.subplots(rows, cols, figsize=(4*cols, 4*rows))
    if rows == 1 and cols == 1:
        axes = np.array([[axes]])
    elif rows == 1:
        axes = axes.reshape(1, -1)
    elif cols == 1:
        axes = axes.reshape(-1, 1)

    for i in range(n_show):
        row, col = i // cols, i % cols
        sample_index = anomaly_list[i]['sample_index']
        reason = anomaly_list[i].get('reason', 'Unknown')

        try:
            image_path = os.path.join(ORIGINAL_TRAIN_DIR, sample_index)
            image = cv2.imread(image_path)
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

            axes[row, col].imshow(image)
            reason_short = reason[:35] + "..." if len(reason) > 35 else reason
            axes[row, col].set_title(f"{sample_index}\n{reason_short}", fontsize=8)
        except:
            axes[row, col].text(0.5, 0.5, 'Error', ha='center', va='center')

        axes[row, col].axis('off')

    # Hide empty subplots
    for i in range(n_show, rows * cols):
        row, col = i // cols, i % cols
        axes[row, col].axis('off')

    plt.suptitle(f'{title} ({len(anomaly_list)} total)', fontsize=14)
    plt.tight_layout()
    plt.show()

# Visualize
visualize_anomalies(shrek_results, "Detected SHREK Images 👹")
visualize_anomalies(slime_results, "Detected SLIME Images 🤢")

In [ ]:
# Print details of anomalous images
if len(anomalous_results) > 0:
    print("\nAnomalous Images Details:")
    print("="*60)

    for i, r in enumerate(anomalous_results[:20]):
        print(f"{i+1}.{r['sample_index']} (ID: {r['sample_id']})")
        print(f"   Type: {r['anomaly_type']}")
        print(f"   Reason: {r['reason']}")
        if r['stats']:
            print(f"   Stats: {r['stats']}")

    if len(anomalous_results) > 20:
        print(f"\n...and {len(anomalous_results) - 20} more")

## 3.Create Cleaned Dataset

In [ ]:
# Create clean dataset directory structure
print("Creating clean dataset directory structure...")

if os.path.exists(CLEAN_DATASET_DIR):
    print(f"Removing existing clean dataset: {CLEAN_DATASET_DIR}")
    shutil.rmtree(CLEAN_DATASET_DIR)

os.makedirs(CLEAN_TRAIN_DIR, exist_ok=True)
os.makedirs(CLEAN_TEST_DIR, exist_ok=True)

print(f"Created: {CLEAN_DATASET_DIR}")
print(f"Created: {CLEAN_TRAIN_DIR}")
print(f"Created: {CLEAN_TEST_DIR}")

In [ ]:
# Get list of valid sample indices
anomalous_indices = set(r['sample_index'] for r in anomalous_results)
valid_indices = [idx for idx in sample_indices if idx not in anomalous_indices]

print(f"Valid images to copy: {len(valid_indices)}")
print(f"Anomalous images to skip: {len(anomalous_indices)}")

In [ ]:
def copy_image_and_mask(sample_index, src_dir, dst_dir):
    """Copy image and its corresponding mask."""
    src_img = os.path.join(src_dir, sample_index)
    dst_img = os.path.join(dst_dir, sample_index)
    shutil.copy2(src_img, dst_img)

    mask_name = sample_index.replace('img_', 'mask_')
    src_mask = os.path.join(src_dir, mask_name)
    dst_mask = os.path.join(dst_dir, mask_name)

    if os.path.exists(src_mask):
        shutil.copy2(src_mask, dst_mask)

    return True


def copy_file_parallel(args):
    """Wrapper for parallel file copying."""
    sample_index, src_dir, dst_dir = args
    try:
        copy_image_and_mask(sample_index, src_dir, dst_dir)
        return True
    except Exception as e:
        print(f"Error copying {sample_index}: {e}")
        return False

In [ ]:
# Copy valid training images (parallel)
print(f"Copying {len(valid_indices)} valid training images...")

copy_args = [(idx, ORIGINAL_TRAIN_DIR, CLEAN_TRAIN_DIR) for idx in valid_indices]

success_count = 0
with ThreadPoolExecutor(max_workers=NUM_WORKERS) as executor:
    futures = [executor.submit(copy_file_parallel, args) for args in copy_args]

    for future in tqdm(as_completed(futures), total=len(futures), desc="Copying train"):
        if future.result():
            success_count += 1

print(f"Successfully copied: {success_count}/{len(valid_indices)} images")

In [ ]:
# Copy test data (unchanged)
print("\nCopying test data...")

test_files = os.listdir(ORIGINAL_TEST_DIR)
print(f"Test files to copy: {len(test_files)}")

for filename in tqdm(test_files, desc="Copying test"):
    src = os.path.join(ORIGINAL_TEST_DIR, filename)
    dst = os.path.join(CLEAN_TEST_DIR, filename)
    shutil.copy2(src, dst)

print(f"Copied {len(test_files)} test files")

In [ ]:
# Create filtered labels CSV
print("\nCreating filtered labels CSV...")

clean_labels_df = original_labels_df[original_labels_df['sample_index'].isin(valid_indices)].reset_index(drop=True)
clean_labels_df.to_csv(CLEAN_LABELS_CSV, index=False)

print(f"Original labels: {len(original_labels_df)}")
print(f"Clean labels: {len(clean_labels_df)}")
print(f"Saved to: {CLEAN_LABELS_CSV}")

print("\nClean dataset class distribution:")
print(clean_labels_df['label'].value_counts())

In [ ]:
# Save removed images list
print("\nSaving removed images list...")

removed_df = pd.DataFrame(anomalous_results)

# Add label information
label_map = dict(zip(original_labels_df['sample_index'], original_labels_df['label']))
removed_df['original_label'] = removed_df['sample_index'].map(label_map)

# Convert stats dict to string
removed_df['stats'] = removed_df['stats'].apply(lambda x: json.dumps(x) if x else '{}')

removed_df.to_csv(REMOVED_CSV, index=False)
print(f"Saved {len(removed_df)} removed images to: {REMOVED_CSV}")

In [ ]:
# Save metadata
metadata = {
    'created_at': datetime.now().isoformat(),
    'detection_version': 'enhanced_v2',
    'original_dataset': ORIGINAL_DATASET_DIR,
    'clean_dataset': CLEAN_DATASET_DIR,
    'original_train_count': len(original_labels_df),
    'clean_train_count': len(clean_labels_df),
    'removed_count': len(anomalous_results),
    'removed_breakdown': {
        'shrek': len(shrek_results),
        'slime': len(slime_results),
        'error': len(error_results)
    },
    'false_positive_overrides': list(FALSE_POSITIVE_IDS),
    'class_distribution': clean_labels_df['label'].value_counts().to_dict(),
    'test_files_count': len(test_files),
    'detection_params': {
        'slime': {
            'fluo_h_min': 40, 'fluo_h_max': 85,
            'fluo_s_min': 60, 'fluo_v_min': 180,
            'min_fluo_frac': 0.01
        },
        'shrek': {
            'pink_guard': 0.10,
            'min_pink_frac': 0.04,
            'min_outlier_frac': 0.10,
            'strong_cold_frac': 0.20,
            'strong_warm_frac': 0.14
        }
    }
}

metadata_path = os.path.join(CLEAN_DATASET_DIR, 'cleaning_metadata.json')
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"Metadata saved to: {metadata_path}")

## 4. Verification

In [ ]:
# Verify clean dataset
print("="*60)
print("VERIFICATION")
print("="*60)

clean_train_files = os.listdir(CLEAN_TRAIN_DIR)
clean_test_files = os.listdir(CLEAN_TEST_DIR)
clean_train_images = [f for f in clean_train_files if f.startswith('img_')]
clean_train_masks = [f for f in clean_train_files if f.startswith('mask_')]

print(f"\nClean train directory:")
print(f"  Images: {len(clean_train_images)}")
print(f"  Masks: {len(clean_train_masks)}")

print(f"\nClean test directory:")
print(f"  Files: {len(clean_test_files)}")

# Verify labels CSV
verified_labels = pd.read_csv(CLEAN_LABELS_CSV)
print(f"\nClean labels CSV:")
print(f"  Rows: {len(verified_labels)}")
print(f"  Columns: {list(verified_labels.columns)}")

# Check all images in labels exist
missing_images = [idx for idx in verified_labels['sample_index'] if idx not in clean_train_images]

if missing_images:
    print(f"\n⚠️ Missing images: {len(missing_images)}")
else:
    print(f"\n✅ All images in labels exist in train directory")

# Verify removed images CSV
verified_removed = pd.read_csv(REMOVED_CSV)
print(f"\nRemoved images CSV:")
print(f"  Rows: {len(verified_removed)}")

In [ ]:
# Final summary
print("\n" + "="*60)
print("✅ DATASET CLEANING COMPLETE! ")
print("="*60)

print(f"\n📁 Clean Dataset Location:")
print(f"   {CLEAN_DATASET_DIR}")

print(f"\n📊 Statistics:")
print(f"   Original samples: {len(original_labels_df)}")
print(f"   Clean samples: {len(clean_labels_df)}")
print(f"   Removed: {len(anomalous_results)} ({100*len(anomalous_results)/len(original_labels_df):.1f}%)")
print(f"     - Shrek: {len(shrek_results)}")
print(f"     - Slime: {len(slime_results)}")
print(f"     - Errors: {len(error_results)}")

print(f"\n📄 Files Created:")
print(f"   • {CLEAN_LABELS_CSV}")
print(f"   • {REMOVED_CSV}")
print(f"   • {metadata_path}")

print(f"\n🔧 Usage in Training Notebook:")
print(f"   # Just change the data path:")
print(f"   data_dir = '{CLEAN_DATASET_DIR}'")
print(f"   train_data_dir = os.path.join(data_dir, 'train_data')")
print(f"   labels_csv = os.path.join(data_dir, 'train_labels.csv')")
print(f"   # No need to scan for anomalies - already cleaned!")

In [ ]:
# Show sample images from clean dataset
print("\nSample images from clean dataset:")

sample_clean = clean_labels_df.sample(min(8, len(clean_labels_df)))

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i, (_, row) in enumerate(sample_clean.iterrows()):
    sample_index = row['sample_index']
    label = row['label']

    image_path = os.path.join(CLEAN_TRAIN_DIR, sample_index)
    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    axes[i].imshow(image)
    axes[i].set_title(f"{sample_index}\n{label}", fontsize=9)
    axes[i].axis('off')

plt.suptitle('Sample Images from Clean Dataset', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(CLEAN_DATASET_DIR, 'sample_clean_images.png'), dpi=150)
plt.show()

## 5. Quick Load Test

In [ ]:
# Test loading the clean dataset
print("Testing clean dataset loading...")

CLEAN_DATA_DIR = "/gdrive/MyDrive/AN2DL_Chall2/Dataset_Clean_v2"

clean_train_dir = os.path.join(CLEAN_DATA_DIR, "train_data")
clean_test_dir = os.path.join(CLEAN_DATA_DIR, "test_data")
clean_labels_csv = os.path.join(CLEAN_DATA_DIR, "train_labels.csv")

# Load labels
df = pd.read_csv(clean_labels_csv)
print(f"✅ Loaded {len(df)} training samples")
print(f"\nClass distribution:")
print(df['label'].value_counts())

# Test loading a random image
random_sample = df.sample(1).iloc[0]
test_image_path = os.path.join(clean_train_dir, random_sample['sample_index'])
test_image = cv2.imread(test_image_path)
print(f"\n✅ Successfully loaded test image: {random_sample['sample_index']}")
print(f"   Shape: {test_image.shape}")
print(f"   Label: {random_sample['label']}")

print("\n" + "="*60)
print("Clean dataset is ready for use!")
print("="*60)